**This notebook on multi-objective optimization provides an overview of the Non-dominated Sorting Genetic Algorithm II (NSGA-II) alongside practical implementation steps ranging from benchmark problem solving to real-world portfolio optimization.**

# **Outline**

1. Installation and library setup
2. Introduction to NSGA-II
3. Using NSGA-II to solve benchmark problem
4. Parameter experiment with NSGA-II
5. Defining a custom multi-objective problem
6. Constrained multi-objective optimization
7. Portfolio Optimization Problem
8. Introduction to NSGA-III

**References**:
- K. Deb, A. Pratap, S. Agarwal and T. Meyarivan, "A fast and elitist multiobjective genetic algorithm: NSGA-II," in IEEE Transactions on Evolutionary Computation, vol. 6, no. 2, pp. 182-197, April 2002, doi: 10.1109/4235.996017.
- K. Deb and H. Jain, "An Evolutionary Many-Objective Optimization Algorithm Using Reference-Point-Based Nondominated Sorting Approach, Part I: Solving Problems With Box Constraints," in IEEE Transactions on Evolutionary Computation, vol. 18, no. 4, pp. 577-601, Aug. 2014, doi: 10.1109/TEVC.2013.2281535.
- Blank, J., & Deb, K. (2020). pymoo: Multi-objective Optimization in Python. IEEE Access.
- Markowitz, H. (1952). Portfolio Selection. The Journal of Finance, 7(1), 77–91.

# 1. Installation

In this tutorial, we will use `pymoo`, a Python library for optimization.
pymoo supports single-objective, multi-objective, many-objective, constrained, and engineering optimization problems. For more information, readers are directed to pymoo's documentation [pymoo documentation](https://pymoo.org/).




# 2. Non-Dominated Sorting Genetic Algorithm II

NSGA-II, or Non-dominated Sorting Genetic Algorithm II, is one of the most widely used evolutionary algorithms for solving multi-objective optimization problems. The main idea of NSGA-II is to evolve a population of candidate solutions while maintaining both convergence toward the Pareto front and diversity among solutions.


NSGA-II ranks solutions using `non-dominated sorting`. Solutions that are not dominated by any other solution are placed in the first front, called $F_1$. After removing $F_1$, the next set of non-dominated solutions forms $F_2$, and so on. **A lower rank indicates a better solution**. The algorithm also uses `crowding distance` to preserve diversity among solutions. In pymoo, NSGA-II uses non-dominated rank and crowding distance to maintain diversity in the objective space, and the crowding distance is based on the Manhattan distance in objective space.

For a solution (i), the crowding distance can be expressed as:

$$
CD_i =
\sum_{m=1}^{M}
\frac{
f_m^{i+1} - f_m^{i-1}
}{
f_m^{\max} - f_m^{\min}
}
$$

where $CD_i$ is the crowding distance of solution $i$, $M$ is the number of objectives, $f_m^{i+1}$ and $f_m^{i-1}$ are the neighboring objective values after sorting based on objective $m$, and $f_m^{\max}$ and $f_m^{\min}$ are the maximum and minimum values of objective $m$.

If two solutions have the same rank, the **solution with a larger crowding distance is preferred** because it helps maintain diversity.

The basic workflow of NSGA-II consists of:
1. initializing a population,
2. evaluating objective functions,
3. sorting solutions into non-dominated fronts,
4. calculating crowding distance,
5. selecting parents,
6. applying `crossover` and `mutation`,
7. combining parent and offspring populations, and
8. selecting the best individuals for the next generation.

Through this process, NSGA-II gradually approximates the Pareto front.



# 3. First Example: Solving ZDT1 with NSGA-II

The benchmark problem used here is **ZDT1**, which is one of the well-known Zitzler-Deb-Thiele test problems for evaluating multi-objective evolutionary algorithms. In `pymoo`, ZDT1 is implemented as a 30-variable problem with two objective functions and a convex Pareto-optimal front.

The ZDT1 problem can be written as:

$$
\min_{\mathbf{x}} \quad \mathbf{F}(\mathbf{x}) =
\left[
f_1(\mathbf{x}), f_2(\mathbf{x})
\right]
$$

where:

$$
f_1(\mathbf{x}) = x_1
$$

$$
f_2(\mathbf{x}) = g(\mathbf{x})h(f_1,g)
$$

$$
g(\mathbf{x}) =
1 + \frac{9}{n-1}
\sum_{i=2}^{n} x_i
$$

$$
h(f_1,g) =
1 - \sqrt{\frac{f_1}{g}}
$$

with the variable bounds:

$$
0 \leq x_i \leq 1,
\quad i = 1,2,\dots,n
$$

For ZDT1, the Pareto-optimal solution occurs when:

$$
0 \leq x_1^* \leq 1
$$

and

$$
x_i^* = 0,
\quad i = 2,3,\dots,n
$$

From the plot, a good optimization result should show points that are close to the true Pareto front and well distributed along the curve. Closeness to the true Pareto front indicates good convergence, while a well-spread distribution of points indicates good diversity.

# 4. Parameter Experiment with NSGA-II

For a population-based evolutionary algorithm such as NSGA-II, the approximate number of function evaluations can be written as:

$$
N_{\mathrm{eval}} \approx P \times G
$$

where:

$$
P = \text{population size}
$$

and

$$
G = \text{number of generations}
$$

If the number of offspring is explicitly defined, the approximate number of evaluations can be written as:

$$
N_{\mathrm{eval}} \approx P + N_{\mathrm{offspring}}(G-1)
$$

where:

$$
N_{\mathrm{offspring}} = \text{number of offspring generated in each generation}
$$

In this experiment, several parameters can be edited:

| Parameter | Meaning |
|---|---|
| `pop_size` | Number of candidate solutions in each generation |
| `n_gen` | Number of generations used as the stopping criterion |
| `n_offsprings` | Number of offspring generated per generation |
| `crossover_prob` | Probability of applying crossover |
| `crossover_eta` | Distribution index for Simulated Binary Crossover, SBX |
| `mutation_eta` | Distribution index for Polynomial Mutation |


In general, a larger `pop_size` may improve the diversity of solutions, while a larger `n_gen` may improve convergence. However, both settings increase computational cost.

To compare the quality of each experiment, we use the Inverted Generational Distance, or IGD. IGD measures how far the obtained solution set is from the reference Pareto front. A smaller IGD value indicates a better approximation of the Pareto front.

In [ ]:
# Import the required modules for the parameter experiment
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM

# Import the performance evaluation matrics
from pymoo.indicators.igd import IGD

# Define the benchmark problem and its true Pareto front.
problem = get_problem("zdt1")
pf = problem.pareto_front()

igd = IGD(pf)

In [ ]:
experiment_settings = [
    {
        "name": "Small population, short run",
        "pop_size": 50,
        "n_gen": 100,
        "n_offsprings": 50,
        "crossover_prob": 0.9,
        "crossover_eta": 15,
        "mutation_eta": 20,
        "seed": 1,
    },
    {
        "name": "Small population, long run",
        "pop_size": 50,
        "n_gen": 200,
        "n_offsprings": 50,
        "crossover_prob": 0.9,
        "crossover_eta": 15,
        "mutation_eta": 20,
        "seed": 1,
    },
    {
        "name": "Large population, short run",
        "pop_size": 100,
        "n_gen": 100,
        "n_offsprings": 100,
        "crossover_prob": 0.9,
        "crossover_eta": 15,
        "mutation_eta": 20,
        "seed": 1,
    },
    {
        "name": "Large population, long run",
        "pop_size": 100,
        "n_gen": 200,
        "n_offsprings": 100,
        "crossover_prob": 0.9,
        "crossover_eta": 15,
        "mutation_eta": 20,
        "seed": 1,
    }
]

A smaller Inverted Generational Distance (IGD) indicates a better approximation of the true Pareto front. As shown in the table, increasing both the population size and the number of generations yields a smaller IGD, thereby improving the approximation closer to the true Pareto front.

# 5. Defining a Custom Multi-objective Problem

We will define our own multi-objective optimization problem instead of using a built-in benchmark problem. This is useful because real optimization problems usually have objective functions that come from a specific application, model, or engineering system.

The custom problem used in this section has two decision variables:

$$
\mathbf{x} = [x_1, x_2]
$$

and two objective functions:

$$
\min_{\mathbf{x}} \quad \mathbf{F}(\mathbf{x}) =
\left[
f_1(\mathbf{x}), f_2(\mathbf{x})
\right]
$$

where:

$$
f_1(\mathbf{x}) = x_1^2 + x_2^2
$$

and:

$$
f_2(\mathbf{x}) = (x_1 - 1)^2 + x_2^2
$$

The variable bounds are:

$$
-2 \leq x_1 \leq 2
$$

$$
-2 \leq x_2 \leq 2
$$

The first objective, $f_1$, is minimized near the point $(0,0)$. The second objective, $f_2$, is minimized near the point $(1,0)$. Because these two objectives have different preferred locations, the optimizer must find a set of trade-off solutions between them.

In `pymoo`, a custom problem can be defined by creating a class that inherits from `ElementwiseProblem`. The main editable parts of the problem definition are:

| Parameter or Component | Meaning |
|---|---|
| `n_var` | Number of decision variables |
| `n_obj` | Number of objective functions |
| `xl` | Lower bound of each decision variable |
| `xu` | Upper bound of each decision variable |
| `_evaluate()` | Function where the objective values are calculated |
| `out["F"]` | Output container for the objective function values |

## 5.1. Challenge #1: Solve this Multiobjective Problem

$$
\mathbf{x} = [x_1, x_2, x_3, x_4]
$$

$$
\min_{\mathbf{x}} \ \mathbf{F}(\mathbf{x}) =
\left[ f_1(\mathbf{x}), f_2(\mathbf{x}), f_3(\mathbf{x}) \right]
$$

$$
f_1(\mathbf{x}) = x_1^2 + x_2^2 + x_3^2 + x_4^2
$$

$$
f_2(\mathbf{x}) =
(x_1 - 1)^2 + (x_2 + 1)^2 + (x_3 - 0.5)^2 + x_4^2
$$

$$
f_3(\mathbf{x}) =
(x_1 + 1)^2 + (x_2 - 1)^2 + (x_3 + 0.5)^2 + (x_4 - 1)^2
$$

$$
-3 \leq x_1 \leq 3
$$

$$
-3 \leq x_2 \leq 3
$$

$$
-3 \leq x_3 \leq 3
$$

$$
-3 \leq x_4 \leq 3
$$

# 6. Constrained Multi-objective Optimization

We extend the custom multi-objective problem by adding a constraint. Constraints are important because real optimization problems usually have limitations, such as physical limits, design requirements, cost limits, safety limits, or operational boundaries.

The constrained problem used in this section has two decision variables:

$$
\mathbf{x} = [x_1, x_2]
$$

and two objective functions:

$$
\min_{\mathbf{x}} \quad \mathbf{F}(\mathbf{x}) =
\left[
f_1(\mathbf{x}), f_2(\mathbf{x})
\right]
$$

where:

$$
f_1(\mathbf{x}) = x_1^2 + x_2^2
$$

and:

$$
f_2(\mathbf{x}) = (x_1 - 1)^2 + x_2^2
$$

The variable bounds are:

$$
-2 \leq x_1 \leq 2
$$

$$
-2 \leq x_2 \leq 2
$$

In this constrained version, we add one inequality constraint:

$$
g_1(\mathbf{x}) = x_1 + x_2 - 0.6 \leq 0
$$

This constraint means that the sum of $x_1$ and $x_2$ must not be greater than 0.6:

$$
x_1 + x_2 \leq 0.6
$$

In `pymoo`, inequality constraints must be written in the form:

$$
g_i(\mathbf{x}) \leq 0
$$

Therefore, the constraint value is stored in:

$$
\texttt{out["G"]}
$$

The main editable parts in this section are:

| Parameter or Component | Meaning |
|---|---|
| `limit` | Constraint limit in $$x_1 + x_2 \leq \text{limit}$$ |
| `n_ieq_constr` | Number of inequality constraints |
| `xl` | Lower bound of each decision variable |
| `xu` | Upper bound of each decision variable |
| `out["F"]` | Objective function values |
| `out["G"]` | Inequality constraint values |
| `pop_size` | Number of candidate solutions in each generation |
| `n_gen` | Number of generations |
| `seed` | Random seed for reproducibility |

A feasible solution must satisfy all inequality constraints. If:

$$
g_1(\mathbf{x}) \leq 0
$$

then the solution is feasible. If:

$$
g_1(\mathbf{x}) > 0
$$

then the solution is infeasible.

## 6.1 Challenge #2: Solve this constrainted multi-objective problem
$$
\mathbf{x} = [x_1, x_2]
$$

$$
\min_{\mathbf{x}} \ \mathbf{F}(\mathbf{x}) =
\left[ f_1(\mathbf{x}), f_2(\mathbf{x}) \right]
$$

where:

$$
f_1(\mathbf{x}) = x_1^2 + x_2^2
$$

$$
f_2(\mathbf{x}) = (x_1 - 1)^2 + x_2^2
$$

subject to:

$$
- x_2 \geq x_1 - 0.8
$$

$$
\frac{5}{4}x_1 - x_2 \geq 0.3
$$

with variable bounds:

$$
-2 \leq x_1 \leq 2
$$

$$
-2 \leq x_2 \leq 2
$$

# 7. Real-World Problem: Portfolio Optimization

In this section, we apply multi-objective optimization to a real-world portfolio optimization problem. Historical stock price data are downloaded using `yfinance`, and NSGA-II is used to find a set of portfolio allocations that balance return and risk.

In portfolio optimization, the decision variables are the portfolio weights:

$$
\mathbf{w} = [w_1, w_2, \dots, w_n]
$$

where $w_i$ is the proportion of capital allocated to asset $i$, and $n$ is the number of assets.

A basic long-only portfolio must satisfy:

$$
0 \leq w_i \leq 1
$$

and:

$$
\sum_{i=1}^{n} w_i = 1
$$

The expected portfolio return is calculated as:

$$
R_p = \mathbf{w}^{T}\boldsymbol{\mu}
$$

where $\boldsymbol{\mu}$ is the vector of annualized expected returns.

The portfolio risk is calculated using the covariance matrix of asset returns:

$$
\sigma_p = \sqrt{\mathbf{w}^{T}\boldsymbol{\Sigma}\mathbf{w}}
$$

where $\boldsymbol{\Sigma}$ is the annualized covariance matrix.

Because `pymoo` solves minimization problems by default, maximizing return is converted into minimizing negative return:

$$
f_1(\mathbf{w}) = -R_p
$$

The second objective is to minimize portfolio risk:

$$
f_2(\mathbf{w}) = \sigma_p
$$

Therefore, the multi-objective portfolio optimization problem is written as:

$$
\min_{\mathbf{w}} \quad \mathbf{F}(\mathbf{w}) =
\left[
-R_p, \sigma_p
\right]
$$

subject to:

$$
\sum_{i=1}^{n} w_i = 1
$$

$$
0 \leq w_i \leq 1
$$

In this tutorial, the weight-sum constraint is handled by normalizing the decision variables:

$$
w_i = \frac{x_i}{\sum_{j=1}^{n} x_j}
$$

This ensures that:

$$
\sum_{i=1}^{n} w_i = 1
$$

We also add a maximum allocation constraint so that no asset dominates the portfolio:

$$
w_i \leq w_{\max}
$$

which is written in `pymoo` inequality form as:

$$
g_i(\mathbf{w}) = w_i - w_{\max} \leq 0
$$

The main editable parameters in this section are:

| Parameter or Component | Meaning |
|---|---|
| `tickers` | List of stock symbols used in the portfolio |
| `period` | Historical data period downloaded from Yahoo Finance |
| `interval` | Data interval, such as daily price data |
| `trading_days` | Number of trading days used for annualization |
| `max_weight` | Maximum allowed allocation for each asset |
| `pop_size` | Number of candidate portfolios in each generation |
| `n_gen` | Number of generations |
| `seed` | Random seed for reproducibility |
| `risk_free_rate` | Risk-free rate used for Sharpe ratio calculation |

The result of this optimization is not a single portfolio. Instead, NSGA-II produces a set of Pareto-optimal portfolios. Each portfolio represents a different trade-off between maximizing return and minimizing risk.

In [ ]:
# Install yfinance
!pip install -U yfinance

# 8. DTLZ1 Problem

The next benchmark problem is **DTLZ1**, which is one of the Deb-Thiele-Laumanns-Zitzler test problems for multi-objective optimization. Unlike ZDT1, which has two objective functions, DTLZ1 is commonly used for problems with three or more objectives.

In this example, DTLZ1 is solved as a **3-objective minimization problem** using NSGA-II.

The DTLZ1 problem can be written as:

$$
\min_{\mathbf{x}} \; \mathbf{F}(\mathbf{x})
=
[f_1(\mathbf{x}), f_2(\mathbf{x}), \ldots, f_M(\mathbf{x})]
$$

where:

$$
f_1(\mathbf{x})
=
\frac{1}{2}x_1x_2 \cdots x_{M-1}(1+g(\mathbf{x}))
$$

$$
f_2(\mathbf{x})
=
\frac{1}{2}x_1x_2 \cdots (1-x_{M-1})(1+g(\mathbf{x}))
$$

$$
\vdots
$$

$$
f_M(\mathbf{x})
=
\frac{1}{2}(1-x_1)(1+g(\mathbf{x}))
$$

The function $$g(\mathbf{x})$$ is defined using the last $$k=n-M+1$$ decision variables:

$$
g(\mathbf{x})
=
100
\left[
k
+
\sum_{i=M}^{n}
\left(
(x_i-0.5)^2
-
\cos(20\pi(x_i-0.5))
\right)
\right]
$$

with the variable bounds:

$$
0 \leq x_i \leq 1,
\qquad
i=1,2,\ldots,n
$$

For DTLZ1, the Pareto-optimal solution occurs when:

$$
x_i^* = 0.5,
\qquad
i=M,M+1,\ldots,n
$$

which gives:

$$
g(\mathbf{x}) = 0
$$

Therefore, the Pareto front of DTLZ1 is a linear hyper-plane:

$$
\sum_{m=1}^{M} f_m = \frac{1}{2},
\qquad
f_m \geq 0
$$

In this example, we use:

$$
M = 3
$$

and

$$
n = M+k-1 = 3+5-1 = 7
$$

Thus, the problem has **7 decision variables** and **3 objective functions**.

## Solving DTLZ1 with NSGA - III

NSGA-III, or Non-dominated Sorting Genetic Algorithm III, is an extension of NSGA-II designed mainly for many-objective optimization problems.

The main idea of NSGA-III is very similar to NSGA-II: it still uses **non-dominated sorting** to rank solutions into Pareto fronts. The key difference is how diversity is maintained.

In **NSGA-II**, diversity is maintained using **crowding distance**.

In **NSGA-III**, diversity is maintained using **reference directions** instead of crowding distance.

Reference directions are predefined directions spread across the objective space. Each solution is associated with the closest reference direction, and solutions are selected so that the population remains well distributed along these directions.

For a solution with objective vector $\mathbf{F}(\mathbf{X})$ and a reference direction $\mathbf{z}$, the perpendicular distance to the reference direction can be written as:

$$
d_2(\mathbf{X}, \mathbf{z})
=
\left\|
\mathbf{F}(\mathbf{X})
-
\frac{\mathbf{z}^T \mathbf{F}(\mathbf{X})}{\|\mathbf{z}\|^2}
\mathbf{z}
\right\|_2
$$

where $d_2$ measures how far the solution is from the reference direction.

Therefore, the main difference can be summarized as:

$$
\text{NSGA-II: non-dominated sorting + crowding distance}
$$

$$
\text{NSGA-III: non-dominated sorting + reference directions}
$$

Because of this, NSGA-III is usually preferred when the number of objectives is large.